In [1]:
import pandas as pd
import sqlite3

# Load the CPIH inflation data
cpih = pd.read_csv('CPIH.csv')

# Load real wages data with correct encoding
wages = pd.read_csv('WAGES.csv', skiprows=8, encoding='latin1')

# Quick peek at both
print("CPIH shape:", cpih.shape)
print(cpih.head())
print("\nWages shape:", wages.shape)
print(wages.head())

CPIH shape: (55510, 7)
    v4_0  mmm-yy  ... cpih1dim1aggid                              Aggregate
0  136.5  Nov-25  ...          CP042     04.2 Owner Occupiers Housing Costs
1  174.6  Nov-25  ...          CP045  04.5 Electricity, gas and other fuels
2  140.0  Nov-25  ...          CP062              06.2 Out-patient services
3  160.2  Nov-25  ...          CP063                 06.3 Hospital services
4  124.2  Nov-25  ...          CP071              07.1 Purchase of vehicles

[5 rows x 7 columns]

Wages shape: (317, 15)
  Unnamed: 0  Unnamed: 1  Unnamed: 2  ...  Unnamed: 12  Unnamed: 13  Unnamed: 14
0     Jan 00       424.0        87.8  ...          NaN          NaN          NaN
1     Feb 00       414.0        85.8  ...          NaN          NaN          NaN
2     Mar 00       429.0        88.9  ...          NaN          NaN          NaN
3     Apr 00       426.0        88.3  ...          NaN          NaN          NaN
4     May 00       430.0        89.1  ...          NaN          NaN   

In [2]:
#  CLEAN THE CPIH DATA 

# Filter to only the Overall Index (CP00) and only the columns needed
cpih_clean = cpih[cpih['cpih1dim1aggid'] == 'CP00'][['mmm-yy', 'v4_0']].copy()

# Rename columns to something meaningful
cpih_clean.columns = ['date', 'cpih_index']

# Convert date column to proper datetime format
cpih_clean['date'] = pd.to_datetime(cpih_clean['date'], format='%b-%y')

# Filter to 2019-2024
cpih_clean = cpih_clean[
    (cpih_clean['date'] >= '2019-01-01') & 
    (cpih_clean['date'] <= '2024-12-31')
].sort_values('date').reset_index(drop=True)

print("CPIH cleaned shape:", cpih_clean.shape)
print(cpih_clean.head(10))

CPIH cleaned shape: (72, 2)
        date  cpih_index
0 2019-01-01       106.4
1 2019-02-01       106.8
2 2019-03-01       107.0
3 2019-04-01       107.6
4 2019-05-01       107.9
5 2019-06-01       107.9
6 2019-07-01       108.0
7 2019-08-01       108.3
8 2019-09-01       108.4
9 2019-10-01       108.3


In [3]:
# CLEAN THE WAGES DATA 

# Keep only the columns needed - that means stopping before footnotes at row 313
wages_clean = wages.iloc[:313, [0, 1, 5]].copy()

# Rename columns
wages_clean.columns = ['date', 'real_awe_total', 'real_awe_regular']

# Standardise date format - replace dashes with spaces
wages_clean['date'] = wages_clean['date'].astype(str).str.replace('-', ' ')

# Parse dates
wages_clean['date'] = pd.to_datetime(wages_clean['date'], format='%b %y')

# Filter to 2019-2024
wages_clean = wages_clean[
    (wages_clean['date'] >= '2019-01-01') &
    (wages_clean['date'] <= '2024-12-31')
].sort_values('date').reset_index(drop=True)

print("Wages cleaned shape:", wages_clean.shape)
print(wages_clean.head(10))

Wages cleaned shape: (72, 3)
        date  real_awe_total  real_awe_regular
0 2019-01-01           494.0             467.0
1 2019-02-01           495.0             465.0
2 2019-03-01           496.0             467.0
3 2019-04-01           500.0             468.0
4 2019-05-01           501.0             469.0
5 2019-06-01           502.0             471.0
6 2019-07-01           503.0             470.0
7 2019-08-01           502.0             471.0
8 2019-09-01           503.0             470.0
9 2019-10-01           502.0             472.0


In [4]:
# --- MERGE BOTH DATASETS ---

# Join on the date column
combined = pd.merge(cpih_clean, wages_clean, on='date', how='inner')

# Calculate purchasing power - wages relative to inflation
# Base: use Jan 2019 as the reference point (100)
base_cpih = combined.loc[combined['date'] == '2019-01-01', 'cpih_index'].values[0]
base_wage = combined.loc[combined['date'] == '2019-01-01', 'real_awe_total'].values[0]

combined['purchasing_power_index'] = (combined['real_awe_total'] / base_wage) * 100
combined['inflation_index'] = (combined['cpih_index'] / base_cpih) * 100

print("Combined shape:", combined.shape)
print(combined.head(10))

Combined shape: (72, 6)
        date  cpih_index  ...  purchasing_power_index  inflation_index
0 2019-01-01       106.4  ...              100.000000       100.000000
1 2019-02-01       106.8  ...              100.202429       100.375940
2 2019-03-01       107.0  ...              100.404858       100.563910
3 2019-04-01       107.6  ...              101.214575       101.127820
4 2019-05-01       107.9  ...              101.417004       101.409774
5 2019-06-01       107.9  ...              101.619433       101.409774
6 2019-07-01       108.0  ...              101.821862       101.503759
7 2019-08-01       108.3  ...              101.619433       101.785714
8 2019-09-01       108.4  ...              101.821862       101.879699
9 2019-10-01       108.3  ...              101.619433       101.785714

[10 rows x 6 columns]


In [5]:
# LOAD INTO SQLite FOR SQL QUERYING 

conn = sqlite3.connect(':memory:')

combined.to_sql('purchasing_power', conn, if_exists='replace', index=False)

# Verify it loaded
result = pd.read_sql_query("SELECT * FROM purchasing_power LIMIT 5", conn)
print(result)

                  date  cpih_index  ...  purchasing_power_index  inflation_index
0  2019-01-01 00:00:00       106.4  ...              100.000000       100.000000
1  2019-02-01 00:00:00       106.8  ...              100.202429       100.375940
2  2019-03-01 00:00:00       107.0  ...              100.404858       100.563910
3  2019-04-01 00:00:00       107.6  ...              101.214575       101.127820
4  2019-05-01 00:00:00       107.9  ...              101.417004       101.409774

[5 rows x 6 columns]


In [6]:
q1 = pd.read_sql_query("""
    SELECT 
        strftime('%Y', date) AS year,
        ROUND(AVG(cpih_index), 2) AS avg_cpih,
        ROUND(AVG(inflation_index), 2) AS avg_inflation_index,
        ROUND(MAX(inflation_index) - MIN(inflation_index), 2) AS inflation_rise_within_year
    FROM purchasing_power
    GROUP BY year
    ORDER BY year
""", conn)

print(q1)

   year  avg_cpih  avg_inflation_index  inflation_rise_within_year
0  2019    107.80               101.32                        1.97
1  2020    108.87               102.32                        1.03
2  2021    111.61               104.89                        5.06
3  2022    120.45               113.20                       10.06
4  2023    128.63               120.90                        5.36
5  2024    132.84               124.85                        4.79


In [7]:
q2 = pd.read_sql_query("""
    SELECT 
        strftime('%Y', date) AS year,
        ROUND(AVG(inflation_index), 2) AS avg_inflation_index,
        ROUND(AVG(purchasing_power_index), 2) AS avg_purchasing_power,
        ROUND(AVG(purchasing_power_index) - AVG(inflation_index), 2) AS gap
    FROM purchasing_power
    GROUP BY year
    ORDER BY year
""", conn)

print(q2)

   year  avg_inflation_index  avg_purchasing_power    gap
0  2019               101.32                101.16  -0.15
1  2020               102.32                102.07  -0.24
2  2021               104.89                105.41   0.52
3  2022               113.20                102.72 -10.49
4  2023               120.90                102.41 -18.48
5  2024               124.85                105.16 -19.69


In [8]:
q3 = pd.read_sql_query("""
    WITH monthly_gap AS (
        SELECT 
            date,
            ROUND(inflation_index, 2) AS inflation_index,
            ROUND(purchasing_power_index, 2) AS purchasing_power_index,
            ROUND(purchasing_power_index - inflation_index, 2) AS gap,
            LAG(ROUND(purchasing_power_index - inflation_index, 2)) 
                OVER (ORDER BY date) AS prev_gap
        FROM purchasing_power
    )
    SELECT 
        date,
        inflation_index,
        purchasing_power_index,
        gap,
        prev_gap
    FROM monthly_gap
    WHERE gap < 0 AND (prev_gap >= 0 OR prev_gap IS NULL)
    ORDER BY date
""", conn)

print(q3)

                  date  inflation_index  purchasing_power_index   gap  prev_gap
0  2019-02-01 00:00:00           100.38                  100.20 -0.17      0.00
1  2019-08-01 00:00:00           101.79                  101.62 -0.17      0.32
2  2021-09-01 00:00:00           105.64                  105.47 -0.17      0.31


In [9]:
q4 = pd.read_sql_query("""
    SELECT 
        date,
        ROUND(inflation_index, 2) AS inflation_index,
        ROUND(purchasing_power_index, 2) AS purchasing_power_index,
        ROUND(purchasing_power_index - inflation_index, 2) AS gap
    FROM purchasing_power
    ORDER BY gap ASC
    LIMIT 5
""", conn)

print(q4)

                  date  inflation_index  purchasing_power_index    gap
0  2024-11-01 00:00:00           126.50                  105.87 -20.63
1  2024-12-01 00:00:00           126.97                  106.68 -20.29
2  2024-10-01 00:00:00           126.22                  106.07 -20.15
3  2024-08-01 00:00:00           125.38                  105.26 -20.11
4  2024-06-01 00:00:00           125.00                  105.06 -19.94


In [10]:
q5 = pd.read_sql_query("""
    WITH monthly_changes AS (
        SELECT
            date,
            ROUND(purchasing_power_index, 2) AS purchasing_power_index,
            ROUND(inflation_index, 2) AS inflation_index,
            ROUND(purchasing_power_index - inflation_index, 2) AS gap,
            ROUND(purchasing_power_index - LAG(purchasing_power_index) 
                OVER (ORDER BY date), 2) AS wage_momentum
        FROM purchasing_power
    )
    SELECT
        strftime('%Y', date) AS year,
        ROUND(AVG(gap), 2) AS avg_gap,
        ROUND(AVG(wage_momentum), 2) AS avg_monthly_wage_growth
    FROM monthly_changes
    GROUP BY year
    ORDER BY year
""", conn)

print(q5)

   year  avg_gap  avg_monthly_wage_growth
0  2019    -0.15                     0.07
1  2020    -0.24                     0.39
2  2021     0.52                     0.07
3  2022   -10.49                    -0.44
4  2023   -18.48                     0.17
5  2024   -19.69                     0.30
